# Evaluate HTv2/BTC Checkpoints with `.lab` Export and Robustness Stress Tests

This notebook evaluates **your own checkpoint**, not the pretrained Jiang repo model.

What it does:
- loads an existing `HTv2` or `BTC` checkpoint from this repo,
- runs framewise inference on `processed/*.npz`,
- exports predictions as `.lab` files,
- scores them with `mir_eval` chord metrics,
- scores large-vocabulary `frame_acc` / `class_acc` using the same seen-only class averaging as the teammate notebook,
- optionally applies **signal decay** or **feature-space Gaussian noise** before inference.

Important constraints:
- no retraining is required,
- this notebook expects `label_mode` to be `full_chord` or `structured_full_chord`,
- robustness is applied to the **processed CQT features**, because this model consumes `processed/*.npz`, not raw audio.


In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path.cwd().resolve()
if REPO_DIR.name == "notebooks":
    REPO_DIR = REPO_DIR.parent
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Repo dir: {REPO_DIR}")


In [ ]:
# Optional: install notebook-only dependencies into the current environment.
# Skip if these packages are already available in your conda env.

import sys
# !{sys.executable} -m pip install -q numpy pandas tqdm mir_eval jams matplotlib seaborn


In [ ]:
from pathlib import Path
import math
import torch

ROOT_DIR = (REPO_DIR / "data" / "chord_data_1217").resolve()
CHECKPOINT_PATH = ROOT_DIR / "checkpoints" / "YOUR_EXPERIMENT" / "fold_0_best.pt"
MODEL_TYPE = "btc"  # "btc" or "htv2"
# Example checkpoint path: ROOT_DIR / "checkpoints" / "compare_btc_fold0" / "fold_0_best.pt"
LABEL_MODE = "structured_full_chord"  # "full_chord" or "structured_full_chord"
FOLD_ID = 0
SPLIT = "test"  # "train", "val", "test", or "all"

# These hyperparameters must match the checkpoint you trained.
EMBED_SIZE = 256
N_LAYERS = 4
N_HEADS = 8
DROPOUT = 0.3
N_STEPS = 128
STRIDE = 64

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WINDOW_BATCH_SIZE = 32
OVERWRITE_PREDICTIONS = False
LIMIT_SONGS = None  # set to a small int for a quick smoke test

# Corruption presets. Choose one mode.
EVAL_MODE = "signal_decay"  # "signal_decay", "feature_noise_snr", or "feature_noise_std"
if EVAL_MODE == "signal_decay":
    LEVELS = {
        "clean": 1.0,
        "little": 0.85,
        "some": 0.60,
        "lots": 0.35,
    }
elif EVAL_MODE == "feature_noise_snr":
    LEVELS = {
        "clean": math.inf,
        "little": 20.0,
        "some": 10.0,
        "lots": 0.0,
    }
elif EVAL_MODE == "feature_noise_std":
    LEVELS = {
        "clean": 0.0,
        "little": 0.01,
        "some": 0.03,
        "lots": 0.08,
    }
else:
    raise ValueError(f"Unsupported EVAL_MODE: {EVAL_MODE}")

OUTPUT_ROOT = (REPO_DIR / "eval_outputs" / CHECKPOINT_PATH.stem / EVAL_MODE / f"fold_{FOLD_ID}_{SPLIT}").resolve()
REF_DIR = (ROOT_DIR / "chordlab").resolve()

assert ROOT_DIR.exists(), f"Missing root dir: {ROOT_DIR}"
assert CHECKPOINT_PATH.exists(), f"Missing checkpoint: {CHECKPOINT_PATH}"
assert REF_DIR.exists(), f"Missing reference lab dir: {REF_DIR}"
assert LABEL_MODE in {"full_chord", "structured_full_chord"}

print(f"Root dir: {ROOT_DIR}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Device: {DEVICE}")
print(f"Mode: {EVAL_MODE} -> {LEVELS}")


In [ ]:
from evaluation_utils import (
    EvalCheckpointConfig,
    apply_feature_noise_snr,
    apply_feature_noise_std,
    apply_signal_decay,
    evaluate_large_vocabulary,
    evaluate_mir_eval_metrics,
    export_predictions,
    load_eval_model,
    resolve_song_ids,
)

cfg = EvalCheckpointConfig(
    root_dir=str(ROOT_DIR),
    checkpoint_path=str(CHECKPOINT_PATH),
    model_type=MODEL_TYPE,
    label_mode=LABEL_MODE,
    fold_id=FOLD_ID,
    split=SPLIT,
    embed_size=EMBED_SIZE,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    dropout=DROPOUT,
    n_steps=N_STEPS,
    stride=STRIDE,
    device=DEVICE,
)

model, vocab, device = load_eval_model(cfg)
print(type(model).__name__)
print(f"Vocab size: {vocab.size}")
print(f"Songs in split: {len(resolve_song_ids(str(ROOT_DIR), FOLD_ID, SPLIT))}")


## Export predictions

Each level writes `.lab` files into `eval_outputs/.../<level>/`.

`signal_decay` multiplies the feature sequence by a linear envelope from `1.0` to `final_gain`.
`feature_noise_snr` and `feature_noise_std` add Gaussian noise directly to the processed CQT features.


In [ ]:
from pathlib import Path

summaries = {}

for level_name, level_value in LEVELS.items():
    level_dir = OUTPUT_ROOT / level_name

    def corruption_fn(x, song_id, *, _mode=EVAL_MODE, _value=level_value):
        if _mode == "signal_decay":
            return x if _value == 1.0 else apply_signal_decay(x, final_gain=_value)
        seed = abs(hash((song_id, level_name))) % (2**32)
        if _mode == "feature_noise_snr":
            return apply_feature_noise_snr(x, snr_db=_value, seed=seed)
        if _mode == "feature_noise_std":
            return apply_feature_noise_std(x, noise_std=_value, seed=seed)
        raise ValueError(_mode)

    if level_name == "clean":
        corruption = None
    else:
        corruption = corruption_fn

    summary = export_predictions(
        cfg=cfg,
        output_dir=str(level_dir),
        corruption_fn=corruption,
        batch_size=WINDOW_BATCH_SIZE,
        overwrite=OVERWRITE_PREDICTIONS,
        limit=LIMIT_SONGS,
    )
    summaries[level_name] = summary
    print(level_name, summary)


## Score exported `.lab` files

`frame_acc` and `class_acc` below use the same **seen-only class averaging** as the teammate notebook.

Note: when chord identities are canonicalized with `mir_eval`, the effective vocabulary key count can be below 301 because enharmonic spellings collapse. That is expected here and matches the notebook-style scorer behavior.


In [ ]:
import pandas as pd

rows = []
vocab_labels = list(vocab.idx_to_chord)

for level_name in LEVELS:
    est_dir = OUTPUT_ROOT / level_name
    mir_scores = evaluate_mir_eval_metrics(str(est_dir), str(REF_DIR))
    large_vocab = evaluate_large_vocabulary(
        est_dir=str(est_dir),
        ref_dir=str(REF_DIR),
        vocab_labels=vocab_labels,
        fps=22050.0 / 512.0,
    )
    rows.append({
        "level": level_name,
        "model_type": MODEL_TYPE,
        "label_mode": LABEL_MODE,
        "frame_acc": large_vocab["frame_acc"],
        "class_acc": large_vocab["class_acc"],
        "classes_seen": large_vocab["classes_seen"],
        "vocab_keys": large_vocab["vocab_key_count"],
        **mir_scores,
    })

results_df = pd.DataFrame(rows)
results_df = results_df[[
    "level",
    "model_type",
    "label_mode",
    "frame_acc",
    "class_acc",
    "classes_seen",
    "vocab_keys",
    "root",
    "thirds",
    "majmin",
    "triads",
    "sevenths",
    "tetrads",
    "mirex",
]]
results_df


In [ ]:
# Optional: save the summary table.
results_csv = OUTPUT_ROOT / "results.csv"
results_df.to_csv(results_csv, index=False)
print(f"Saved: {results_csv}")
